In [1]:
import numpy as np
import pandas as pd
import glob

# Read files in

In [2]:
files = glob.glob("data/spcsv/*.csv")

dfs = []
for f in files:
    df = pd.read_csv(f)

    df.columns = (df.columns.str.strip().str.lower())
    dfs.append(df)

df = pd.concat(dfs, ignore_index=True)
df = df.dropna()

# Normalise data

## Assign location

In [3]:
df["location"] = (df.menu_hint
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.split(" ")
    .str[2:-2]
    .str.join(" "))
df = df[df.location != ""]

## Fix poorly formatted runner names

In [4]:
df.selection_name = df.selection_name.str.replace(r"^\d+\.? ", "", regex=True)

## Distance

In [5]:
df = df[df.event_name.str.contains(r"^\d")] # remove poor formatting
df["distance_f"] = (df.event_name.str.strip().str.extract(r"(?:(\d+)m)?(?:(\d)f?)?").fillna(0).astype(int) * [8, 1]).sum(axis=1)
df.loc[(df.distance_f == 0) | (df.distance_f >= 40), "distance_f"] = np.nan

## Race type

In [6]:
df["race_type"] = df.event_name.str.strip().str.split().str[1:].str.join(" ")

## Date

In [7]:
df["race_time"] = pd.to_datetime(df.event_dt, dayfirst=True)
df["event_date"] = df.race_time.dt.date

## Assign event IDs

In [8]:
df["race_id"] = df["event_id"]
df["event_id"] = df.groupby(["location", "event_date"]).ngroup()

## Implied probability

In [9]:
df["implied_odds"] = 1/df.bsp

## Remove redundancies

In [10]:
df = df.drop(columns=["menu_hint", "event_name", "event_dt"])
df = df.dropna()

## Assign types

In [11]:
df = df.astype({
    "event_id": "int64",
    "selection_id": "int64",
    "selection_name": "string",
    "win_lose": "bool",
    "bsp": "float64",
    "ppwap": "float32",
    "morningwap": "float32",
    "ppmax": "float32",
    "ppmin": "float32",
    "ipmax": "float32",
    "ipmin": "float32",
    "morningtradedvol": "float32",
    "pptradedvol": "float32",
    "iptradedvol": "float32",
    "location": "string",
    "distance_f": "int8",
    "race_type": "string",
    "race_time": "datetime64[s]",
    "event_date": "datetime64[s]",
    "race_id": "int64",
    "implied_odds": "float64"
})

# Observe result

In [12]:
df

,event_id,selection_id,selection_name,win_lose,bsp,ppwap,morningwap,ppmax,ppmin,ipmax,...,morningtradedvol,pptradedvol,iptradedvol,location,distance_f,race_type,race_time,event_date,race_id,implied_odds
0,1535,8709395,Zylan,False,23.000000,18.397112,10.189857,22.00,9.24,1000.0,...,642.739990,13388.459961,1743.660034,Bev,7,Hcap,2019-09-24 15:05:00,2019-09-24,162764294,0.043478
1,1535,20377712,Major Snugfit,False,7.837297,7.288927,7.290506,9.40,6.00,1000.0,...,1088.199951,58811.339844,7166.410156,Bev,10,Hcap,2019-09-24 17:45:00,2019-09-24,162764320,0.127595
2,1535,24209475,Bosuns Chair,False,150.000000,156.691589,90.800003,300.00,42.00,1000.0,...,65.540001,2000.260010,508.010010,Bev,5,Nov Stks,2019-09-24 14:00:00,2019-09-24,162764284,0.006667
3,1535,24209478,Dancinginthesand,False,32.000000,24.183567,22.995310,34.00,19.50,1000.0,...,277.100006,6302.540039,1921.500000,Bev,5,Nov Stks,2019-09-24 14:00:00,2019-09-24,162764284,0.031250
4,2924,24148082,Taamer,False,6.200000,7.027211,8.440805,9.60,5.80,1000.0,...,631.780029,32564.160156,4578.720215,ChelmC,7,Nov Stks,2019-09-24 17:25:00,2019-09-24,162764331,0.161290
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1611324,18084,40484789,Alyas Gold Award,False,7.380804,7.965647,6.534649,10.50,7.00,1000.0,...,615.000000,33865.320312,4295.209961,Windsor,6,Nov Stks,2021-08-09 18:05:00,2021-08-09,186161169,0.135487
1611325,18084,40262716,Dansemai,False,17.441746,19.368006,40.297195,37.57,16.50,1000.0,...,24.900000,10423.820312,2537.709961,Windsor,6,Nov Stks,2021-08-09 18:05:00,2021-08-09,186161169,0.057334
1611326,19307,35415057,Chase That Dream,False,4.963830,3.770604,3.324620,5.60,3.00,1000.0,...,18485.019531,115489.703125,5235.740234,Wolverhampton,7,Hcap,2021-08-09 13:00:00,2021-08-09,186161205,0.201457
1611327,19307,28625715,John Jasper,False,24.258500,15.513046,12.547174,27.00,10.00,1000.0,...,1025.699951,14853.299805,2449.949951,Wolverhampton,8,Hcap,2021-08-09 13:35:00,2021-08-09,186161210,0.041223


## Events

In [13]:
events = df[["event_id", "location", "event_date"]].drop_duplicates().set_index("event_id")
events

,location,event_date
event_id,,
1535,Bev,2019-09-24
2924,ChelmC,2019-09-24
9908,Ling,2019-09-24
17242,Warw,2019-09-24
8589,Leic,2011-03-31
...,...,...
10874,Muss,2017-05-04
803,Ayr,2021-08-09
14184,Ripon,2021-08-09


## Races

In [14]:
races = df[["race_id", "race_type", "race_time", "distance_f", "event_id"]].drop_duplicates().set_index("race_id")
races

,race_type,race_time,distance_f,event_id
race_id,,,,
162764294,Hcap,2019-09-24 15:05:00,7,1535
162764320,Hcap,2019-09-24 17:45:00,10,1535
162764284,Nov Stks,2019-09-24 14:00:00,5,1535
162764331,Nov Stks,2019-09-24 17:25:00,7,2924
162764361,Hcap,2019-09-24 20:30:00,14,2924
...,...,...,...,...
186160695,Nov Stks,2021-08-09 15:35:00,7,803
186161070,Hcap,2021-08-09 19:55:00,5,14184
186161164,App Hcap,2021-08-09 17:35:00,6,18084


## Runners

In [15]:
runners = df[[
    "selection_id",
    "race_id",
    "selection_name",
    "win_lose",
    "bsp",
    "implied_odds",
    "ppwap",
    "morningwap",
    "ppmax",
    "ppmin",
    "ipmax",
    "ipmin",
    "morningtradedvol",
    "pptradedvol",
    "iptradedvol"
]].set_index("selection_id")
runners

,race_id,selection_name,win_lose,bsp,implied_odds,ppwap,morningwap,ppmax,ppmin,ipmax,ipmin,morningtradedvol,pptradedvol,iptradedvol
selection_id,,,,,,,,,,,,,,
8709395,162764294,Zylan,False,23.000000,0.043478,18.397112,10.189857,22.00,9.24,1000.0,12.00,642.739990,13388.459961,1743.660034
20377712,162764320,Major Snugfit,False,7.837297,0.127595,7.288927,7.290506,9.40,6.00,1000.0,2.90,1088.199951,58811.339844,7166.410156
24209475,162764284,Bosuns Chair,False,150.000000,0.006667,156.691589,90.800003,300.00,42.00,1000.0,21.00,65.540001,2000.260010,508.010010
24209478,162764284,Dancinginthesand,False,32.000000,0.031250,24.183567,22.995310,34.00,19.50,1000.0,8.60,277.100006,6302.540039,1921.500000
24148082,162764331,Taamer,False,6.200000,0.161290,7.027211,8.440805,9.60,5.80,1000.0,4.30,631.780029,32564.160156,4578.720215
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40484789,186161169,Alyas Gold Award,False,7.380804,0.135487,7.965647,6.534649,10.50,7.00,1000.0,7.20,615.000000,33865.320312,4295.209961
40262716,186161169,Dansemai,False,17.441746,0.057334,19.368006,40.297195,37.57,16.50,1000.0,5.50,24.900000,10423.820312,2537.709961
35415057,186161205,Chase That Dream,False,4.963830,0.201457,3.770604,3.324620,5.60,3.00,1000.0,3.65,18485.019531,115489.703125,5235.740234


# Store

In [17]:
events.to_parquet("data/events.parquet")
races.to_parquet("data/races.parquet")
runners.to_parquet("data/runners.parquet")